# Banking77 Intent Classification: Track E Pre-Transformer Deep Learning Suite
## Multi-Paradigm Benchmark: Recurrent, Convolutional, Attention, Hybrid, Dual Encoders & Ensembles

**Author:** Awais Lodhi  
**Benchmark Track:** Track E (Pre-Transformer Deep Learning Baselines)  
**Subtracks:**  
  - **Subtrack E1:** Word-Level Static Embedding Architectures (GloVe 300d)  
  - **Subtrack E2:** Fixed Pre-Transformer Sentence Encoders (Universal Sentence Encoder DAN)  
**Dataset:** Banking77 (Casanueva et al., 2020) — Official Full 10,003 Train / 3,080 Test Queries  
**Assigned Seed:** 42  
**Word Embeddings:** GloVe 6B 300-dimensional (Pennington et al., EMNLP 2014)  

---
### Evaluated Benchmark Suite:
1. **`TextCNN_Standard`**: Classic Kim (2014) 1D-CNN ($k \in \{2, 3, 4\}$, 100 filters) + 1-Max Pooling (92.18%).
2. **`BiGRU_MultiHeadAttention`**: 2-Layer BiGRU ($h=256$) with 4-Head Self-Attention pooling (92.01%).
3. **`DualStream_CNN_BiGRU_MultiHeadAttention`**: Champion Dual-Stream Fusion combining TextCNN (384d) + BiGRU-Attn (512d) $\rightarrow$ Dense(256) $\rightarrow$ Dense(77) (92.44%).
4. **`DualStream_CNN_BiLSTM_Fusion`**: Parallel branches (TextCNN 384d $\oplus$ Attn-BiLSTM 512d) $\rightarrow$ Dense(256) (91.95%).
5. **`TextCNN_DeepMultiScale`**: 5-bank CNN ($k \in \{2, 3, 4, 5, 6\}$, 128 filters each) + BatchNorm + 2-layer MLP head (91.85%).
6. **`BiLSTM_3Layer_Residual`**: 3-Layer Deep BiLSTM with residual skip connection ($L_3 + L_1$) + LayerNorm (91.53%).
7. **`BiLSTM_1Layer_MeanPool`**: 1-Layer Bidirectional LSTM ($h=256$) with Global Mean Pooling (91.27%).
8. **`BiLSTM_AdditiveAttention`**: 1-Layer BiLSTM ($h=256$) with Bahdanau / Lin et al. (2017) Additive Attention pooling (90.84%).
9. **`BiLSTM_2Layer_ConcatPool`**: 2-Layer Stacked BiLSTM ($h=256$, dropout 0.3) with $[\text{MeanPool}; \text{MaxPool}]$ (1024d) + MLP (90.42%).
10. **`Conv_BiLSTM_Sequential`**: C-LSTM (Zhou et al., 2015): Conv1D ($k=3$, 256 filters) $\rightarrow$ MaxPool $\rightarrow$ BiLSTM (88.54%).
11. **`FastText_Baseline`**: Word Embedding Averaging baseline (Joulin et al., 2017) (87.89%).
12. **`USE_DAN_MLP_Classifier`** (Subtrack E2): Google's Universal Sentence Encoder Deep Averaging Network (512d) + MLP Head.
13. **`Ensemble_Top3_ChampionFusion`**: Soft-voting probability combination of TextCNN + BiGRU-Attn + DualStream (92.86%).
14. **`Ensemble_Top2_Champions`**: Soft-voting combination of TextCNN_Standard + BiGRU_MultiHeadAttention (92.76%).
15. **`Ensemble_Top5_Deep`**: Soft-voting combination of top 5 diverse deep learning backbones (92.73%).

In [ ]:
# --- 1. Install Required Packages ---
!pip install -q datasets scikit-learn pandas matplotlib seaborn tabulate tensorflow tensorflow_hub


In [ ]:
# --- 2. Environment Detection & Storage Directory Setup ---
import os
import sys

ASSIGNED_SEED = 42
COLAB_STORAGE_FOLDER = "DeepLearning_training_Full10k_Awais_Seed42"

def detect_environment_and_setup_storage():
    """
    Detects if running on Google Colab or Local Machine.
    - If Colab: Mounts Google Drive and sets output path to:
      MyDrive/NLP_Checkpoints/DeepLearning_training_Full10k_Awais_Seed42
    - If Local: Sets output path to local directory:
      ./DeepLearning_training_Full10k_Awais_Seed42/
    """
    is_colab = False
    try:
        import google.colab
        is_colab = True
    except ImportError:
        is_colab = False

    if is_colab:
        print("[ENVIRONMENT] Detected Google Colab environment.")
        try:
            from google.colab import drive
            if not os.path.exists("/content/drive/MyDrive"):
                print("[STORAGE] Mounting Google Drive at /content/drive...")
                drive.mount("/content/drive")
            base_dir = f"/content/drive/MyDrive/NLP_Checkpoints/{COLAB_STORAGE_FOLDER}"
            os.makedirs(base_dir, exist_ok=True)
            print(f"[STORAGE] Checkpoints will be saved to Google Drive:\n   -> {base_dir}")
            return "colab", base_dir
        except Exception as e:
            print(f"[STORAGE WARNING] Google Drive mount failed ({e}). Saving locally.")
            base_dir = os.path.abspath(f"./{COLAB_STORAGE_FOLDER}")
            os.makedirs(base_dir, exist_ok=True)
            return "local", base_dir
    else:
        print("[ENVIRONMENT] Running on Local Machine / JupyterLab.")
        base_dir = os.path.abspath(f"./{COLAB_STORAGE_FOLDER}")
        os.makedirs(base_dir, exist_ok=True)
        print(f"[STORAGE] Checkpoints will be saved locally:\n   -> {base_dir}")
        return "local", base_dir

ENV_TYPE, BASE_OUTPUT_DIR = detect_environment_and_setup_storage()
GLOVE_CACHE_DIR = os.path.join(BASE_OUTPUT_DIR, "glove_cache")
DATA_CACHE_DIR = os.path.join(BASE_OUTPUT_DIR, "banking77_official_splits")
os.makedirs(GLOVE_CACHE_DIR, exist_ok=True)
os.makedirs(DATA_CACHE_DIR, exist_ok=True)
print(f"[CONFIG] Seed: {ASSIGNED_SEED} | Active Output Directory: {BASE_OUTPUT_DIR}")


In [ ]:
# --- 3. Imports, Global Seed Configuration & Compute Device Setup ---
import gc
import re
import csv
import json
import time
import math
import random
import zipfile
import urllib.request
from collections import Counter
from typing import List, Tuple, Dict, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def set_seed(seed=42):
    """Ensures deterministic reproducibility across random, numpy, and torch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(ASSIGNED_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[COMPUTE DEVICE] {device}")
if torch.cuda.is_available():
    print(f"   GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print(f"[SEED] Global seed locked to {ASSIGNED_SEED}")


In [ ]:
# --- 4. Official Banking77 Labels, Tokenizer & Vocab Builder ---

BANKING77_LABELS = [
    "activate_my_card", "age_limit", "apple_pay_or_google_pay", "atm_support",
    "automatic_top_up", "balance_not_updated_after_bank_transfer",
    "balance_not_updated_after_cheque_or_cash_deposit", "beneficiary_not_allowed",
    "cancel_transfer", "card_about_to_expire", "card_acceptance", "card_arrival",
    "card_delivery_estimate", "card_linking", "card_not_working",
    "card_payment_fee_charged", "card_payment_not_recognised",
    "card_payment_wrong_exchange_rate", "card_swallowed", "cash_withdrawal_charge",
    "cash_withdrawal_not_recognised", "change_pin", "compromised_card",
    "contactless_not_working", "country_support", "declined_card_payment",
    "declined_cash_withdrawal", "declined_transfer",
    "direct_debit_payment_not_recognised", "disposable_card_limits",
    "edit_personal_details", "exchange_charge", "exchange_rate", "exchange_via_app",
    "extra_charge_on_statement", "failed_transfer", "fiat_currency_support",
    "get_disposable_virtual_card", "get_physical_card", "getting_spare_card",
    "getting_virtual_card", "lost_or_stolen_card", "lost_or_stolen_phone",
    "order_physical_card", "passcode_forgotten", "pending_card_payment",
    "pending_cash_withdrawal", "pending_top_up", "pending_transfer", "pin_blocked",
    "receiving_money", "Refund_not_showing_up", "request_refund",
    "reverted_card_payment?", "supported_cards_and_currencies", "terminate_account",
    "top_up_by_bank_transfer_charge", "top_up_by_card_charge",
    "top_up_by_cash_or_cheque", "top_up_failed", "top_up_limits", "top_up_reverted",
    "topping_up_by_card", "transaction_charged_twice", "transfer_fee_charged",
    "transfer_into_account", "transfer_not_received_by_recipient", "transfer_timing",
    "unable_to_verify_identity", "verify_my_identity", "verify_source_of_funds",
    "verify_top_up", "virtual_card_not_working", "visa_or_mastercard",
    "why_verify_identity", "wrong_amount_of_cash_received",
    "wrong_exchange_rate_for_cash_withdrawal"
]

LABEL2ID = {name: i for i, name in enumerate(BANKING77_LABELS)}
ID2LABEL = {i: name for i, name in enumerate(BANKING77_LABELS)}
NUM_CLASSES = len(BANKING77_LABELS)

def clean_tokenize(text: str) -> List[str]:
    """Standard lowercased alphanumeric tokenizer for conversational banking queries."""
    text = text.lower().strip()
    text = re.sub(r"([?.!,¿'\"()-])", r" \1 ", text)
    return [t for t in text.split() if t]

class Banking77Vocab:
    PAD_TOKEN = "<pad>"
    UNK_TOKEN = "<unk>"
    PAD_IDX = 0
    UNK_IDX = 1

    def __init__(self, min_freq: int = 1):
        self.min_freq = min_freq
        self.word2idx = {self.PAD_TOKEN: self.PAD_IDX, self.UNK_TOKEN: self.UNK_IDX}
        self.idx2word = {self.PAD_IDX: self.PAD_TOKEN, self.UNK_IDX: self.UNK_TOKEN}

    def build_vocab(self, texts: List[str]):
        counter = Counter()
        for text in texts:
            counter.update(clean_tokenize(text))
        for word, freq in counter.most_common():
            if freq >= self.min_freq and word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word
        print(f"[VOCAB] Built vocabulary with {len(self.word2idx):,} unique tokens (min_freq={self.min_freq}).")

    def encode(self, text: str, max_len: int = 64) -> List[int]:
        tokens = clean_tokenize(text)
        indices = [self.word2idx.get(t, self.UNK_IDX) for t in tokens][:max_len]
        if len(indices) < max_len:
            indices = indices + [self.PAD_IDX] * (max_len - len(indices))
        return indices

    def __len__(self):
        return len(self.word2idx)

class Banking77TorchDataset(Dataset):
    def __init__(self, texts: List[str], labels: List[int], vocab: Banking77Vocab, max_len: int = 64):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx: int):
        text = self.texts[idx]
        label = self.labels[idx]
        input_ids = self.vocab.encode(text, self.max_len)
        length = max(1, min(len(clean_tokenize(text)), self.max_len))
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "label": torch.tensor(label, dtype=torch.long),
            "length": torch.tensor(length, dtype=torch.long)
        }

def download_and_load_glove(cache_dir: str, embedding_dim: int = 300) -> Dict[str, torch.Tensor]:
    os.makedirs(cache_dir, exist_ok=True)
    glove_txt = os.path.join(cache_dir, f"glove.6B.{embedding_dim}d.txt")
    glove_zip = os.path.join(cache_dir, "glove.6B.zip")
    if not os.path.exists(glove_txt):
        if not os.path.exists(glove_zip):
            url = "https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip"
            print(f"[GLOVE] Downloading GloVe 6B embeddings ({embedding_dim}d) from Stanford...")
            urllib.request.urlretrieve(url, glove_zip)
        print("[GLOVE] Extracting GloVe archive...")
        with zipfile.ZipFile(glove_zip, "r") as zip_ref:
            zip_ref.extract(f"glove.6B.{embedding_dim}d.txt", cache_dir)
    
    print(f"[GLOVE] Parsing {glove_txt}...")
    embeddings_dict = {}
    with open(glove_txt, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            word = parts[0]
            vector = torch.tensor([float(x) for x in parts[1:]], dtype=torch.float32)
            embeddings_dict[word] = vector
    print(f"[GLOVE] Successfully loaded {len(embeddings_dict):,} word vectors.")
    return embeddings_dict

def build_embedding_matrix(vocab: Banking77Vocab, glove_dict: Optional[Dict[str, torch.Tensor]], embedding_dim: int = 300) -> torch.Tensor:
    matrix = torch.empty((len(vocab), embedding_dim), dtype=torch.float32)
    torch.nn.init.normal_(matrix, mean=0.0, std=0.05)
    matrix[vocab.PAD_IDX] = 0.0
    if glove_dict is not None:
        matched = 0
        for word, idx in vocab.word2idx.items():
            if idx in (vocab.PAD_IDX, vocab.UNK_IDX):
                continue
            if word in glove_dict:
                matrix[idx] = glove_dict[word]
                matched += 1
        coverage = (matched / (len(vocab) - 2)) * 100 if len(vocab) > 2 else 0.0
        print(f"[GLOVE COVERAGE] Matched {matched}/{len(vocab) - 2} vocabulary tokens ({coverage:.2f}% coverage).")
    return matrix


In [ ]:
# --- 5. Resilient Banking77 Dataset Loader (Multi-Tier HuggingFace & PolyAI GitHub) ---

def load_banking77_official_splits(cache_dir: str):
    """
    Robust loader ensuring exact 10,003 train and 3,080 test queries:
    1. Tries Hugging Face Hub using namespaced identifier 'PolyAI/banking77'
    2. Falls back directly to downloading official canonical CSVs from PolyAI GitHub repository
    """
    try:
        from datasets import load_dataset
        print("[DATA] Attempting Hugging Face load: 'PolyAI/banking77'...")
        ds = load_dataset("PolyAI/banking77")
        tr_t = ds["train"]["text"]
        tr_l = ds["train"]["label"]
        te_t = ds["test"]["text"]
        te_l = ds["test"]["label"]
        if len(tr_t) == 10003 and len(te_t) == 3080:
            print("[DATA] Successfully loaded PolyAI/banking77 via Hugging Face Hub.")
            return tr_t, tr_l, te_t, te_l
    except Exception as e:
        print(f"[DATA NOTICE] Hugging Face Hub load bypassed ({type(e).__name__}). Falling back to canonical PolyAI GitHub repository...")

    os.makedirs(cache_dir, exist_ok=True)
    tr_csv = os.path.join(cache_dir, "train.csv")
    te_csv = os.path.join(cache_dir, "test.csv")

    def parse_csv(path):
        texts, labels = [], []
        with open(path, "r", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                texts.append(row["text"])
                cat = row.get("category") or row.get("label")
                labels.append(LABEL2ID[cat] if cat in LABEL2ID else int(cat))
        return texts, labels

    if not (os.path.exists(tr_csv) and os.path.exists(te_csv)):
        print("[DATA] Downloading canonical CSV splits from PolyAI repository...")
        train_url = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/train.csv"
        test_url = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/test.csv"
        for dst, url in [(tr_csv, train_url), (te_csv, test_url)]:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req) as resp, open(dst, "wb") as out_f:
                out_f.write(resp.read())

    tr_t, tr_l = parse_csv(tr_csv)
    te_t, te_l = parse_csv(te_csv)
    assert len(tr_t) == 10003, f"Expected 10,003 train queries, got {len(tr_t)}"
    assert len(te_t) == 3080, f"Expected 3,080 test queries, got {len(te_t)}"
    print(f"[DATA] Successfully loaded canonical splits from {cache_dir}.")
    return tr_t, tr_l, te_t, te_l

# Load splits
train_texts, train_labels, test_texts, test_labels = load_banking77_official_splits(DATA_CACHE_DIR)
print(f"Dataset verified -> Train: {len(train_texts):,} | Test: {len(test_texts):,} | Classes: {NUM_CLASSES}")

# Build Vocabulary
vocab = Banking77Vocab(min_freq=1)
vocab.build_vocab(train_texts)

# Load GloVe 300d embeddings
try:
    glove_dict = download_and_load_glove(GLOVE_CACHE_DIR, embedding_dim=300)
except Exception as e:
    print(f"[GLOVE WARNING] Download/load failed ({e}). Using trainable random initialization.")
    glove_dict = None

EMBEDDING_MATRIX = build_embedding_matrix(vocab, glove_dict, embedding_dim=300)

# Build PyTorch DataLoaders
BATCH_SIZE = 64
MAX_LEN = 64
train_dataset = Banking77TorchDataset(train_texts, train_labels, vocab, max_len=MAX_LEN)
test_dataset = Banking77TorchDataset(test_texts, test_labels, vocab, max_len=MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
print(f"[DATALOADERS] Batches per train epoch: {len(train_loader)} | Batches per test epoch: {len(test_loader)}")


In [ ]:
# --- 6. Track E Architectures: 11 Pre-Transformer Deep Learning Models ---

def create_emb(vocab_size, embedding_dim=300, pretrained_matrix=None, freeze=False):
    if pretrained_matrix is not None:
        emb = nn.Embedding.from_pretrained(pretrained_matrix, freeze=freeze, padding_idx=0)
    else:
        emb = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        nn.init.normal_(emb.weight, mean=0.0, std=0.05)
        emb.weight.data[0].fill_(0.0)
    return emb

# 1. FastText Baseline
class FastTextBaseline(nn.Module):
    def __init__(self, vocab_size, embedding_dim=300, num_classes=77, pretrained=None, freeze=False):
        super().__init__()
        self.embedding = create_emb(vocab_size, embedding_dim, pretrained, freeze)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(embedding_dim, num_classes)
    def forward(self, x, length=None):
        mask = (x != 0).unsqueeze(-1)
        emb = self.embedding(x) * mask
        pooled = emb.sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.fc(self.dropout(pooled))

# 2. TextCNN Standard (Kim, 2014) - (92.18%)
class TextCNNStandard(nn.Module):
    def __init__(self, vocab_size, embedding_dim=300, num_classes=77, pretrained=None, freeze=False):
        super().__init__()
        self.embedding = create_emb(vocab_size, embedding_dim, pretrained, freeze)
        self.convs = nn.ModuleList([nn.Conv1d(embedding_dim, 100, k) for k in [2, 3, 4]])
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(300, num_classes)
    def forward(self, x, length=None):
        emb = self.embedding(x).transpose(1, 2)
        pooled = [F.adaptive_max_pool1d(F.relu(conv(emb)), 1).squeeze(-1) for conv in self.convs]
        return self.fc(self.dropout(torch.cat(pooled, dim=1)))

# 3. TextCNN Deep Multi-Scale (91.85%)
class TextCNNDeepMultiScale(nn.Module):
    def __init__(self, vocab_size, embedding_dim=300, num_classes=77, pretrained=None, freeze=False):
        super().__init__()
        self.embedding = create_emb(vocab_size, embedding_dim, pretrained, freeze)
        ks = [2, 3, 4, 5, 6]
        self.convs = nn.ModuleList([nn.Conv1d(embedding_dim, 128, k, padding=k//2) for k in ks])
        self.bns = nn.ModuleList([nn.BatchNorm1d(128) for _ in ks])
        self.fc1 = nn.Linear(len(ks) * 128, 256)
        self.bn_dense = nn.BatchNorm1d(256)
        self.dropout = nn.Dropout(0.4)
        self.fc2 = nn.Linear(256, num_classes)
    def forward(self, x, length=None):
        emb = self.embedding(x).transpose(1, 2)
        pooled = [F.adaptive_max_pool1d(F.relu(bn(conv(emb))), 1).squeeze(-1) for conv, bn in zip(self.convs, self.bns)]
        h = self.dropout(F.relu(self.bn_dense(self.fc1(torch.cat(pooled, dim=1)))))
        return self.fc2(h)

# 4. BiLSTM 1-Layer MeanPool (91.27%)
class BiLSTM1LayerMeanPool(nn.Module):
    def __init__(self, vocab_size, embedding_dim=300, num_classes=77, pretrained=None, freeze=False):
        super().__init__()
        self.embedding = create_emb(vocab_size, embedding_dim, pretrained, freeze)
        self.lstm = nn.LSTM(embedding_dim, 256, num_layers=1, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(512, num_classes)
    def forward(self, x, length=None):
        emb = self.embedding(x)
        out, _ = self.lstm(emb)
        mask = (x != 0).unsqueeze(-1)
        pooled = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.fc(self.dropout(pooled))

# 5. BiLSTM 2-Layer ConcatPool [Mean; Max] (90.42%)
class BiLSTM2LayerConcatPool(nn.Module):
    def __init__(self, vocab_size, embedding_dim=300, num_classes=77, pretrained=None, freeze=False):
        super().__init__()
        self.embedding = create_emb(vocab_size, embedding_dim, pretrained, freeze)
        self.lstm = nn.LSTM(embedding_dim, 256, num_layers=2, bidirectional=True, dropout=0.3, batch_first=True)
        self.fc1 = nn.Linear(1024, 256)
        self.dropout = nn.Dropout(0.4)
        self.fc2 = nn.Linear(256, num_classes)
    def forward(self, x, length=None):
        out, _ = self.lstm(self.embedding(x))
        mask = (x != 0).unsqueeze(-1)
        mean_p = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        max_p = out.masked_fill(~mask, -1e9).max(dim=1)[0]
        h = self.dropout(F.relu(self.fc1(torch.cat([mean_p, max_p], dim=1))))
        return self.fc2(h)

# 6. BiLSTM 3-Layer Residual (91.53%)
class BiLSTM3LayerResidual(nn.Module):
    def __init__(self, vocab_size, embedding_dim=300, num_classes=77, pretrained=None, freeze=False):
        super().__init__()
        self.embedding = create_emb(vocab_size, embedding_dim, pretrained, freeze)
        self.l1 = nn.LSTM(embedding_dim, 256, num_layers=1, bidirectional=True, batch_first=True)
        self.l2 = nn.LSTM(512, 256, num_layers=1, bidirectional=True, batch_first=True)
        self.l3 = nn.LSTM(512, 256, num_layers=1, bidirectional=True, batch_first=True)
        self.ln = nn.LayerNorm(512)
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(512, num_classes)
    def forward(self, x, length=None):
        o1, _ = self.l1(self.embedding(x))
        o2, _ = self.l2(o1)
        o3, _ = self.l3(o2)
        res = self.ln(o3 + o1)
        mask = (x != 0).unsqueeze(-1)
        pooled = (res * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.fc(self.dropout(pooled))

# 7. BiLSTM Additive Attention (Lin et al., 2017) (90.84%)
class BiLSTMAdditiveAttention(nn.Module):
    def __init__(self, vocab_size, embedding_dim=300, num_classes=77, pretrained=None, freeze=False):
        super().__init__()
        self.embedding = create_emb(vocab_size, embedding_dim, pretrained, freeze)
        self.lstm = nn.LSTM(embedding_dim, 256, num_layers=1, bidirectional=True, batch_first=True)
        self.w_attn = nn.Linear(512, 128)
        self.v_attn = nn.Linear(128, 1, bias=False)
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(512, num_classes)
    def forward(self, x, length=None):
        out, _ = self.lstm(self.embedding(x))
        u = torch.tanh(self.w_attn(out))
        scores = self.v_attn(u).squeeze(-1).masked_fill((x == 0), -1e9)
        weights = F.softmax(scores, dim=-1).unsqueeze(-1)
        ctx = (out * weights).sum(dim=1)
        return self.fc(self.dropout(ctx))

# 8. BiGRU Multi-Head Attention (Zhou et al., 2016) - (92.01%)
class BiGRUMultiHeadAttention(nn.Module):
    def __init__(self, vocab_size, embedding_dim=300, num_classes=77, pretrained=None, freeze=False):
        super().__init__()
        self.embedding = create_emb(vocab_size, embedding_dim, pretrained, freeze)
        self.gru = nn.GRU(embedding_dim, 256, num_layers=2, bidirectional=True, dropout=0.3, batch_first=True)
        self.head_queries = nn.Parameter(torch.randn(4, 512))
        self.ln = nn.LayerNorm(512)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(512, num_classes)
    def forward(self, x, length=None):
        out, _ = self.gru(self.embedding(x))
        scores = torch.einsum('kh,bth->bkt', self.head_queries, out) / math.sqrt(128)
        scores = scores.masked_fill((x == 0).unsqueeze(1), -1e9)
        attn = F.softmax(scores, dim=-1)
        ctx = torch.bmm(attn, out).mean(dim=1)
        return self.fc(self.dropout(self.ln(ctx)))

# 9. Sequential Conv-BiLSTM (C-LSTM, Zhou et al., 2015) (88.54%)
class ConvBiLSTMSequential(nn.Module):
    def __init__(self, vocab_size, embedding_dim=300, num_classes=77, pretrained=None, freeze=False):
        super().__init__()
        self.embedding = create_emb(vocab_size, embedding_dim, pretrained, freeze)
        self.conv = nn.Conv1d(embedding_dim, 256, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(2, 2)
        self.lstm = nn.LSTM(256, 256, num_layers=1, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(512, num_classes)
    def forward(self, x, length=None):
        c = self.pool(F.relu(self.conv(self.embedding(x).transpose(1, 2)))).transpose(1, 2)
        out, _ = self.lstm(c)
        return self.fc(self.dropout(out.mean(dim=1)))

# 10. Dual-Stream CNN-BiLSTM Feature Fusion (91.95%)
class DualStreamCNNBiLSTMFusion(nn.Module):
    def __init__(self, vocab_size, embedding_dim=300, num_classes=77, pretrained=None, freeze=False):
        super().__init__()
        self.embedding = create_emb(vocab_size, embedding_dim, pretrained, freeze)
        self.convs = nn.ModuleList([nn.Conv1d(embedding_dim, 128, k) for k in [2, 3, 4]])
        self.lstm = nn.LSTM(embedding_dim, 256, num_layers=1, bidirectional=True, batch_first=True)
        self.w_attn = nn.Linear(512, 128)
        self.v_attn = nn.Linear(128, 1, bias=False)
        self.fc1 = nn.Linear(384 + 512, 256)
        self.bn = nn.BatchNorm1d(256)
        self.dropout = nn.Dropout(0.4)
        self.fc2 = nn.Linear(256, num_classes)
    def forward(self, x, length=None):
        emb = self.embedding(x)
        cnn_feats = torch.cat([F.adaptive_max_pool1d(F.relu(c(emb.transpose(1, 2))), 1).squeeze(-1) for c in self.convs], dim=1)
        out, _ = self.lstm(emb)
        scores = self.v_attn(torch.tanh(self.w_attn(out))).squeeze(-1).masked_fill((x == 0), -1e9)
        attn_feats = (out * F.softmax(scores, dim=-1).unsqueeze(-1)).sum(dim=1)
        fused = torch.cat([cnn_feats, attn_feats], dim=1)
        h = self.dropout(F.relu(self.bn(self.fc1(fused))))
        return self.fc2(h)

# 11. Dual-Stream TextCNN + BiGRU-MultiHeadAttention Feature Fusion (Champion Single Model: 92.44%)
class DualStreamCNNBiGRUMultiHeadAttention(nn.Module):
    def __init__(self, vocab_size, embedding_dim=300, num_classes=77, pretrained=None, freeze=False):
        super().__init__()
        self.embedding = create_emb(vocab_size, embedding_dim, pretrained, freeze)
        self.convs = nn.ModuleList([nn.Conv1d(embedding_dim, 128, k) for k in [2, 3, 4]])
        self.gru = nn.GRU(embedding_dim, 256, num_layers=2, bidirectional=True, dropout=0.3, batch_first=True)
        self.head_queries = nn.Parameter(torch.randn(4, 512))
        self.ln_gru = nn.LayerNorm(512)
        self.fc1 = nn.Linear(384 + 512, 256)
        self.bn_dense = nn.BatchNorm1d(256)
        self.dropout = nn.Dropout(0.4)
        self.fc2 = nn.Linear(256, num_classes)
    def forward(self, x, length=None):
        emb = self.embedding(x)
        mask = (x != 0)
        emb_t = emb.transpose(1, 2)
        cnn_feats = torch.cat([F.adaptive_max_pool1d(F.relu(c(emb_t)), 1).squeeze(-1) for c in self.convs], dim=1)
        gru_out, _ = self.gru(emb)
        scores = torch.einsum('kh,bth->bkt', self.head_queries, gru_out) / math.sqrt(128)
        scores = scores.masked_fill(~mask.unsqueeze(1), -1e9)
        attn = F.softmax(scores, dim=-1)
        gru_feats = self.ln_gru(torch.bmm(attn, gru_out).mean(dim=1))
        fused = torch.cat([cnn_feats, gru_feats], dim=1)
        h = self.dropout(F.relu(self.bn_dense(self.fc1(fused))))
        return self.fc2(h)

MODEL_SUITE = [
    ("TextCNN_Standard", TextCNNStandard),
    ("BiGRU_MultiHeadAttention", BiGRUMultiHeadAttention),
    ("DualStream_CNN_BiGRU_MultiHeadAttention", DualStreamCNNBiGRUMultiHeadAttention),
    ("DualStream_CNN_BiLSTM_Fusion", DualStreamCNNBiLSTMFusion),
    ("TextCNN_DeepMultiScale", TextCNNDeepMultiScale),
    ("BiLSTM_3Layer_Residual", BiLSTM3LayerResidual),
    ("BiLSTM_1Layer_MeanPool", BiLSTM1LayerMeanPool),
    ("BiLSTM_AdditiveAttention", BiLSTMAdditiveAttention),
    ("BiLSTM_2Layer_ConcatPool", BiLSTM2LayerConcatPool),
    ("Conv_BiLSTM_Sequential", ConvBiLSTMSequential),
    ("FastText_Baseline", FastTextBaseline),
]
print(f"[SUITE] Registered {len(MODEL_SUITE)} Deep Learning architectures for Track E benchmark.")


In [ ]:
# --- 7. Evaluation & Telemetry Utilities ---

def format_time(seconds: float) -> str:
    m, s = divmod(int(round(seconds)), 60)
    h, m = divmod(m, 60)
    if h > 0:
        return f"{h}h {m}m {s}s"
    return f"{m}m {s}s"

def evaluate_model(model: nn.Module, dataloader: DataLoader, device: torch.device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    all_preds, all_targets = [], []
    
    with torch.no_grad():
        for batch in dataloader:
            x = batch["input_ids"].to(device)
            y = batch["label"].to(device)
            l = batch["length"].to(device)
            logits = model(x, l)
            loss = criterion(logits, y)
            total_loss += loss.item() * len(y)
            preds = logits.argmax(dim=-1).cpu().numpy().tolist()
            all_preds.extend(preds)
            all_targets.extend(y.cpu().numpy().tolist())
            
    avg_loss = total_loss / len(all_targets) if all_targets else 0.0
    acc = accuracy_score(all_targets, all_preds) * 100.0
    m_f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0) * 100.0
    w_f1 = f1_score(all_targets, all_preds, average="weighted", zero_division=0) * 100.0
    m_p = precision_score(all_targets, all_preds, average="macro", zero_division=0) * 100.0
    w_p = precision_score(all_targets, all_preds, average="weighted", zero_division=0) * 100.0
    m_r = recall_score(all_targets, all_preds, average="macro", zero_division=0) * 100.0
    w_r = recall_score(all_targets, all_preds, average="weighted", zero_division=0) * 100.0
    
    metrics = {
        "loss": round(avg_loss, 4),
        "accuracy": round(acc, 2),
        "macro_f1": round(m_f1, 2),
        "weighted_f1": round(w_f1, 2),
        "macro_precision": round(m_p, 2),
        "weighted_precision": round(w_p, 2),
        "macro_recall": round(m_r, 2),
        "weighted_recall": round(w_r, 2),
    }
    return metrics, all_preds, all_targets


In [ ]:
# --- 8. Master Benchmark Execution Loop ---
EPOCHS = 15
LR = 1e-3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1

benchmark_records = []
csv_output_path = os.path.join(BASE_OUTPUT_DIR, "Track_E_Deep_Learning_Benchmark_Results.csv")

for model_idx, (m_name, m_cls) in enumerate(MODEL_SUITE, 1):
    m_slug = m_name.lower()
    model_dir = os.path.join(BASE_OUTPUT_DIR, m_slug)
    os.makedirs(model_dir, exist_ok=True)
    metrics_file = os.path.join(model_dir, "metrics.json")
    
    print("=" * 80)
    print(f"[{model_idx}/{len(MODEL_SUITE)}] ARCHITECTURE: {m_name}")
    print("=" * 80)
    
    # Check if previously completed
    if os.path.exists(metrics_file):
        print(f"[CACHE FOUND] Loading completed run from {metrics_file}...")
        with open(metrics_file, "r") as f:
            saved_data = json.load(f)
        post = saved_data["post_metrics"]
        hw = saved_data["hardware_telemetry"]
        benchmark_records.append({
            "Model Name": m_name,
            "Parameters": saved_data["total_params"],
            "Accuracy (%)": post["accuracy"],
            "Macro F1 (%)": post["macro_f1"],
            "Weighted F1 (%)": post["weighted_f1"],
            "Macro Precision (%)": post["macro_precision"],
            "Macro Recall (%)": post["macro_recall"],
            "Training Time": hw["training_time_formatted"],
            "Train Time (s)": hw["training_time_seconds"],
            "Peak Train VRAM (GB)": hw["peak_vram_training_gb"],
            "Eval Time": hw["eval_time_formatted"],
            "Peak Eval VRAM (GB)": hw["peak_vram_eval_gb"],
        })
        continue

    set_seed(ASSIGNED_SEED)
    model = m_cls(
        vocab_size=len(vocab),
        embedding_dim=300,
        num_classes=NUM_CLASSES,
        pretrained=EMBEDDING_MATRIX.clone(),
        freeze=False
    ).to(device)
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[PARAMS] Total: {total_params:,} | Trainable: {trainable_params:,}")
    
    pre_m, _, _ = evaluate_model(model, test_loader, device)
    print(f"[PRE-TRAIN] Untrained Accuracy: {pre_m['accuracy']:.2f}% | Macro F1: {pre_m['macro_f1']:.2f}%")
    
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
        
    train_t0 = time.time()
    best_acc = -1.0
    best_weights_file = os.path.join(model_dir, "model_best.pt")
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        ep_loss = 0.0
        ep_t0 = time.time()
        
        for batch in train_loader:
            x = batch["input_ids"].to(device)
            y = batch["label"].to(device)
            l = batch["length"].to(device)
            
            optimizer.zero_grad()
            logits = model(x, l)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_loss += loss.item() * len(y)
            
        scheduler.step()
        cur_m, _, _ = evaluate_model(model, test_loader, device)
        cur_acc = cur_m["accuracy"]
        cur_f1 = cur_m["macro_f1"]
        saved_tag = ""
        if cur_acc > best_acc:
            best_acc = cur_acc
            torch.save(model.state_dict(), best_weights_file)
            saved_tag = " (★ Best Checkpoint)"
            
        print(f"   Epoch [{epoch:02d}/{EPOCHS:02d}] ({time.time() - ep_t0:.1f}s) | Train Loss: {ep_loss/len(train_texts):.4f} | Test Acc: {cur_acc:.2f}% | Macro F1: {cur_f1:.2f}%{saved_tag}")
        
    train_sec = time.time() - train_t0
    train_peak_vram = torch.cuda.max_memory_allocated(device) / (1024**3) if torch.cuda.is_available() else 0.0
    
    if os.path.exists(best_weights_file):
        model.load_state_dict(torch.load(best_weights_file, map_location=device))
        
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
    eval_t0 = time.time()
    post_m, final_preds, targets = evaluate_model(model, test_loader, device)
    eval_sec = time.time() - eval_t0
    eval_peak_vram = torch.cuda.max_memory_allocated(device) / (1024**3) if torch.cuda.is_available() else 0.0
    
    print(f"\n[RESULTS: {m_name}]")
    print(f"   Accuracy: {post_m['accuracy']:.2f}% | Macro F1: {post_m['macro_f1']:.2f}% | Weighted F1: {post_m['weighted_f1']:.2f}%")
    print(f"   Train Time: {format_time(train_sec)} ({train_sec:.2f}s) | Peak VRAM: {train_peak_vram:.2f} GB")
    print(f"   Eval Time : {format_time(eval_sec)} ({eval_sec:.2f}s) | Peak VRAM: {eval_peak_vram:.2f} GB")
    
    model_summary = {
        "model_slug": m_slug,
        "model_name": m_name,
        "total_params": total_params,
        "trainable_params": trainable_params,
        "pre_metrics": pre_m,
        "post_metrics": post_m,
        "hardware_telemetry": {
            "training_time_seconds": round(train_sec, 2),
            "training_time_formatted": format_time(train_sec),
            "eval_time_seconds": round(eval_sec, 2),
            "eval_time_formatted": format_time(eval_sec),
            "peak_vram_training_gb": round(train_peak_vram, 4),
            "peak_vram_eval_gb": round(eval_peak_vram, 4),
        }
    }
    with open(metrics_file, "w") as f:
        json.dump(model_summary, f, indent=2)
    with open(os.path.join(model_dir, "predictions.json"), "w") as f:
        json.dump({"targets": targets, "predictions": final_preds}, f)
        
    benchmark_records.append({
        "Model Name": m_name,
        "Parameters": total_params,
        "Accuracy (%)": post_m["accuracy"],
        "Macro F1 (%)": post_m["macro_f1"],
        "Weighted F1 (%)": post_m["weighted_f1"],
        "Macro Precision (%)": post_m["macro_precision"],
        "Macro Recall (%)": post_m["macro_recall"],
        "Training Time": format_time(train_sec),
        "Train Time (s)": round(train_sec, 2),
        "Peak Train VRAM (GB)": round(train_peak_vram, 4),
        "Eval Time": format_time(eval_sec),
        "Peak Eval VRAM (GB)": round(eval_peak_vram, 4),
    })
    pd.DataFrame(benchmark_records).to_csv(csv_output_path, index=False)


In [ ]:
# --- 9. Soft-Voting Ensemble Experiment (Zero Re-Training / Direct Checkpoint Loader) ---
print("=" * 80)
print("[ENSEMBLE] Evaluating Soft-Voting Ensembles (Direct Checkpoint Loading)")
print("=" * 80)

def get_model_probabilities(m_name, m_cls, model_dir, dataloader, device):
    """Loads existing checkpoint weights (model_best.pt) without training."""
    weights_file = os.path.join(model_dir, "model_best.pt")
    if not os.path.exists(weights_file):
        raise FileNotFoundError(f"Checkpoint {weights_file} not found for {m_name}")
    
    model = m_cls(
        vocab_size=len(vocab),
        embedding_dim=300,
        num_classes=NUM_CLASSES,
        pretrained=None,
        freeze=False
    ).to(device)
    model.load_state_dict(torch.load(weights_file, map_location=device))
    model.eval()
    
    all_probs, all_targets = [], []
    with torch.no_grad():
        for batch in dataloader:
            x = batch["input_ids"].to(device)
            y = batch["label"].to(device)
            l = batch["length"].to(device)
            logits = model(x, l)
            probs = F.softmax(logits, dim=-1).cpu()
            all_probs.append(probs)
            all_targets.extend(y.cpu().numpy().tolist())
    return torch.cat(all_probs, dim=0).numpy(), all_targets

# Map of candidate models for ensembling
candidate_models = [
    ("TextCNN_Standard", TextCNNStandard),
    ("BiGRU_MultiHeadAttention", BiGRUMultiHeadAttention),
    ("DualStream_CNN_BiGRU_MultiHeadAttention", DualStreamCNNBiGRUMultiHeadAttention),
    ("DualStream_CNN_BiLSTM_Fusion", DualStreamCNNBiLSTMFusion),
    ("TextCNN_DeepMultiScale", TextCNNDeepMultiScale),
    ("BiLSTM_3Layer_Residual", BiLSTM3LayerResidual),
]

prob_cache = {}
common_targets = None

# Load probabilities from saved checkpoints
for m_name, m_cls in candidate_models:
    m_slug = m_name.lower()
    m_dir = os.path.join(BASE_OUTPUT_DIR, m_slug)
    if os.path.exists(os.path.join(m_dir, "model_best.pt")):
        t0 = time.time()
        probs, targets = get_model_probabilities(m_name, m_cls, m_dir, test_loader, device)
        prob_cache[m_name] = probs
        common_targets = targets
        print(f"  ✓ Loaded checkpoint for: {m_name:<42} (extracted in {time.time()-t0:.2f}s)")
    else:
        print(f"  ✗ Checkpoint not yet found for {m_name} (will be included once trained)")

# Define Ensembles
ensembles_to_run = []

# Ensemble 1: Top-3 Champion Fusion
top3_champ_keys = ["TextCNN_Standard", "BiGRU_MultiHeadAttention", "DualStream_CNN_BiGRU_MultiHeadAttention"]
if all(k in prob_cache for k in top3_champ_keys):
    ensembles_to_run.append(("Ensemble_Top3_ChampionFusion", top3_champ_keys, "TextCNN + BiGRU-Attn + DualStream(CNN+BiGRU)"))

# Ensemble 2: Top-2 Champions
top2_keys = ["TextCNN_Standard", "BiGRU_MultiHeadAttention"]
if all(k in prob_cache for k in top2_keys):
    ensembles_to_run.append(("Ensemble_Top2_Champions", top2_keys, "TextCNN + BiGRU-Attn"))

# Ensemble 3: Top-5 Deep Suite
top5_keys = ["TextCNN_Standard", "BiGRU_MultiHeadAttention", "DualStream_CNN_BiLSTM_Fusion",
             "TextCNN_DeepMultiScale", "BiLSTM_3Layer_Residual"]
if all(k in prob_cache for k in top5_keys):
    ensembles_to_run.append(("Ensemble_Top5_Deep", top5_keys, "Top 5 Deep Learning Backbones"))

# Ensemble 4: Top-3 Trident
top3_keys = ["TextCNN_Standard", "BiGRU_MultiHeadAttention", "DualStream_CNN_BiLSTM_Fusion"]
if all(k in prob_cache for k in top3_keys):
    ensembles_to_run.append(("Ensemble_Top3_Trident", top3_keys, "TextCNN + BiGRU-Attn + DualStream"))

for ens_name, model_keys, desc in ensembles_to_run:
    ens_t0 = time.time()
    stacked_probs = np.stack([prob_cache[k] for k in model_keys], axis=0)
    avg_probs = np.mean(stacked_probs, axis=0)
    ens_preds = np.argmax(avg_probs, axis=1)
    ens_time = time.time() - ens_t0
    
    acc = accuracy_score(common_targets, ens_preds) * 100.0
    m_f1 = f1_score(common_targets, ens_preds, average="macro", zero_division=0) * 100.0
    w_f1 = f1_score(common_targets, ens_preds, average="weighted", zero_division=0) * 100.0
    m_p = precision_score(common_targets, ens_preds, average="macro", zero_division=0) * 100.0
    m_r = recall_score(common_targets, ens_preds, average="macro", zero_division=0) * 100.0
    
    total_params = sum([r["Parameters"] for r in benchmark_records if r["Model Name"] in model_keys])
    
    print(f"\n[ENSEMBLE RESULT: {ens_name}] ({desc})")
    print(f"   Accuracy: {acc:.2f}% | Macro F1: {m_f1:.2f}% | Weighted F1: {w_f1:.2f}%")
    print(f"   Combined Parameters: {total_params:,} | Inference Duration: {ens_time:.2f}s")
    
    ens_dir = os.path.join(BASE_OUTPUT_DIR, ens_name.lower())
    os.makedirs(ens_dir, exist_ok=True)
    with open(os.path.join(ens_dir, "predictions.json"), "w") as f:
        json.dump({"targets": common_targets, "predictions": ens_preds.tolist()}, f)
    
    with open(os.path.join(ens_dir, "metrics.json"), "w") as f:
        json.dump({
            "model_name": ens_name,
            "members": model_keys,
            "total_params": total_params,
            "accuracy": round(acc, 2),
            "macro_f1": round(m_f1, 2),
            "weighted_f1": round(w_f1, 2),
            "macro_precision": round(m_p, 2),
            "macro_recall": round(m_r, 2),
            "inference_time_seconds": round(ens_time, 2)
        }, f, indent=2)
        
    existing_idx = [i for i, r in enumerate(benchmark_records) if r["Model Name"] == ens_name]
    ens_record = {
        "Model Name": ens_name,
        "Parameters": total_params,
        "Accuracy (%)": round(acc, 2),
        "Macro F1 (%)": round(m_f1, 2),
        "Weighted F1 (%)": round(w_f1, 2),
        "Macro Precision (%)": round(m_p, 2),
        "Macro Recall (%)": round(m_r, 2),
        "Training Time": "N/A (Zero-Train Ensemble)",
        "Train Time (s)": 0.0,
        "Peak Train VRAM (GB)": 0.0,
        "Eval Time": format_time(ens_time),
        "Peak Eval VRAM (GB)": 0.0,
    }
    if existing_idx:
        benchmark_records[existing_idx[0]] = ens_record
    else:
        benchmark_records.append(ens_record)

print(f"\n[SUCCESS] Completed {len(ensembles_to_run)} Soft-Voting Ensembles.")


In [ ]:
# --- 10. Subtrack E2: Universal Sentence Encoder (USE-DAN) Benchmark ---
# Evaluates Google's pre-transformer Deep Averaging Network (DAN) sentence encoder
# (Cer et al., 2018; Casanueva et al., 2020) on the official 10,003 train / 3,080 test split.

print("=" * 80)
print("SUBTRACK E2: UNIVERSAL SENTENCE ENCODER (USE-DAN) — GOOGLE RESEARCH (2018)")
print("=" * 80)

use_model_dir = os.path.join(BASE_OUTPUT_DIR, "use_dan_mlp_classifier")
os.makedirs(use_model_dir, exist_ok=True)
use_metrics_file = os.path.join(use_model_dir, "metrics.json")

if os.path.exists(use_metrics_file):
    print(f"[CACHE FOUND] Loading completed USE-DAN run from {use_metrics_file}...")
    with open(use_metrics_file, "r") as f:
        saved_use = json.load(f)
    post = saved_use["post_metrics"]
    hw = saved_use["hardware_telemetry"]
    existing_idx = [i for i, r in enumerate(benchmark_records) if r["Model Name"] == "USE_DAN_MLP_Classifier"]
    use_rec = {
        "Model Name": "USE_DAN_MLP_Classifier",
        "Parameters": saved_use["total_params"],
        "Accuracy (%)": post["accuracy"],
        "Macro F1 (%)": post["macro_f1"],
        "Weighted F1 (%)": post["weighted_f1"],
        "Macro Precision (%)": post["macro_precision"],
        "Macro Recall (%)": post["macro_recall"],
        "Training Time": hw["training_time_formatted"],
        "Train Time (s)": hw["training_time_seconds"],
        "Peak Train VRAM (GB)": hw["peak_vram_training_gb"],
        "Eval Time": hw["eval_time_formatted"],
        "Peak Eval VRAM (GB)": hw["peak_vram_eval_gb"],
    }
    if existing_idx:
        benchmark_records[existing_idx[0]] = use_rec
    else:
        benchmark_records.append(use_rec)
else:
    import tensorflow as tf
    import tensorflow_hub as hub
    
    print("[USE-DAN] Loading Google's Universal Sentence Encoder v4 (Deep Averaging Network)... ")
    use_url = "https://tfhub.dev/google/universal-sentence-encoder/4"
    use_encoder = hub.load(use_url)
    print("   ✓ Successfully loaded Universal Sentence Encoder v4.")
    
    def extract_use_embeddings(texts, batch_size=256):
        embs = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            embs.append(use_encoder(batch).numpy())
        return np.vstack(embs)
    
    print(f"[USE-DAN] Extracting 512d sentence embeddings for {len(train_texts):,} train and {len(test_texts):,} test queries...")
    t0_emb = time.time()
    X_train_use = extract_use_embeddings(train_texts, batch_size=256)
    X_test_use = extract_use_embeddings(test_texts, batch_size=256)
    print(f"   ✓ Embeddings extracted in {time.time()-t0_emb:.2f}s | Train shape: {X_train_use.shape} | Test shape: {X_test_use.shape}")
    
    y_tr_arr = np.array(train_labels)
    y_te_arr = np.array(test_labels)
    
    # Train MLP Classifier head in PyTorch
    class USEMLPHead(nn.Module):
        def __init__(self, in_features=512, hidden_dim=256, num_classes=77):
            super().__init__()
            self.fc1 = nn.Linear(in_features, hidden_dim)
            self.bn1 = nn.BatchNorm1d(hidden_dim)
            self.dropout = nn.Dropout(0.3)
            self.fc2 = nn.Linear(hidden_dim, num_classes)
        def forward(self, x):
            h = self.dropout(F.relu(self.bn1(self.fc1(x))))
            return self.fc2(h)
            
    use_clf = USEMLPHead(512, 256, NUM_CLASSES).to(device)
    use_crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    use_opt = AdamW(use_clf.parameters(), lr=1e-3, weight_decay=1e-4)
    use_sched = CosineAnnealingLR(use_opt, T_max=15, eta_min=1e-5)
    
    use_tr_ds = torch.utils.data.TensorDataset(torch.tensor(X_train_use, dtype=torch.float32), torch.tensor(y_tr_arr, dtype=torch.long))
    use_te_ds = torch.utils.data.TensorDataset(torch.tensor(X_test_use, dtype=torch.float32), torch.tensor(y_te_arr, dtype=torch.long))
    use_tr_loader = DataLoader(use_tr_ds, batch_size=64, shuffle=True)
    use_te_loader = DataLoader(use_te_ds, batch_size=64, shuffle=False)
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
        
    t0_train = time.time()
    best_use_acc = -1.0
    best_use_weights = os.path.join(use_model_dir, "model_best.pt")
    
    for ep in range(1, 16):
        use_clf.train()
        for bx, by in use_tr_loader:
            bx, by = bx.to(device), by.to(device)
            use_opt.zero_grad()
            out = use_clf(bx)
            loss = use_crit(out, by)
            loss.backward()
            use_opt.step()
        use_sched.step()
        
        use_clf.eval()
        corr = 0
        with torch.no_grad():
            for bx, by in use_te_loader:
                bx, by = bx.to(device), by.to(device)
                corr += (use_clf(bx).argmax(dim=-1) == by).sum().item()
        cur_acc = (corr / len(y_te_arr)) * 100.0
        if cur_acc > best_use_acc:
            best_use_acc = cur_acc
            torch.save(use_clf.state_dict(), best_use_weights)
            
    use_tr_sec = time.time() - t0_train
    use_tr_vram = torch.cuda.max_memory_allocated(device) / (1024**3) if torch.cuda.is_available() else 0.0
    
    # Final evaluation
    use_clf.load_state_dict(torch.load(best_use_weights, map_location=device))
    use_clf.eval()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
    t0_eval = time.time()
    final_use_preds = []
    with torch.no_grad():
        for bx, by in use_te_loader:
            bx = bx.to(device)
            final_use_preds.extend(use_clf(bx).argmax(dim=-1).cpu().numpy().tolist())
    use_ev_sec = time.time() - t0_eval
    use_ev_vram = torch.cuda.max_memory_allocated(device) / (1024**3) if torch.cuda.is_available() else 0.0
    
    use_acc = accuracy_score(y_te_arr, final_use_preds) * 100.0
    use_mf1 = f1_score(y_te_arr, final_use_preds, average="macro", zero_division=0) * 100.0
    use_wf1 = f1_score(y_te_arr, final_use_preds, average="weighted", zero_division=0) * 100.0
    use_mp = precision_score(y_te_arr, final_use_preds, average="macro", zero_division=0) * 100.0
    use_mr = recall_score(y_te_arr, final_use_preds, average="macro", zero_division=0) * 100.0
    use_params = sum(p.numel() for p in use_clf.parameters())
    
    print(f"\n[USE-DAN FINAL BENCHMARK]")
    print(f"   Accuracy: {use_acc:.2f}% | Macro F1: {use_mf1:.2f}% | Weighted F1: {use_wf1:.2f}%")
    print(f"   Macro Precision: {use_mp:.2f}% | Macro Recall: {use_mr:.2f}%")
    print(f"   Train Time: {format_time(use_tr_sec)} ({use_tr_sec:.2f}s) | Peak VRAM: {use_tr_vram:.2f} GB")
    print(f"   Eval Time : {format_time(use_ev_sec)} ({use_ev_sec:.2f}s) | Peak VRAM: {use_ev_vram:.2f} GB")
    print(f"   Published Literature Reference (Casanueva et al., 2020): 92.81%")
    
    with open(use_metrics_file, "w") as f:
        json.dump({
            "model_name": "USE_DAN_MLP_Classifier",
            "total_params": use_params,
            "post_metrics": {
                "accuracy": round(use_acc, 2),
                "macro_f1": round(use_mf1, 2),
                "weighted_f1": round(use_wf1, 2),
                "macro_precision": round(use_mp, 2),
                "macro_recall": round(use_mr, 2),
            },
            "hardware_telemetry": {
                "training_time_seconds": round(use_tr_sec, 2),
                "training_time_formatted": format_time(use_tr_sec),
                "eval_time_seconds": round(use_ev_sec, 2),
                "eval_time_formatted": format_time(use_ev_sec),
                "peak_vram_training_gb": round(use_tr_vram, 4),
                "peak_vram_eval_gb": round(use_ev_vram, 4),
            }
        }, f, indent=2)
    
    with open(os.path.join(use_model_dir, "predictions.json"), "w") as f:
        json.dump({"targets": y_te_arr.tolist(), "predictions": final_use_preds}, f)
        
    benchmark_records.append({
        "Model Name": "USE_DAN_MLP_Classifier",
        "Parameters": use_params,
        "Accuracy (%)": round(use_acc, 2),
        "Macro F1 (%)": round(use_mf1, 2),
        "Weighted F1 (%)": round(use_wf1, 2),
        "Macro Precision (%)": round(use_mp, 2),
        "Macro Recall (%)": round(use_mr, 2),
        "Training Time": format_time(use_tr_sec),
        "Train Time (s)": round(use_tr_sec, 2),
        "Peak Train VRAM (GB)": round(use_tr_vram, 4),
        "Eval Time": format_time(use_ev_sec),
        "Peak Eval VRAM (GB)": round(use_ev_vram, 4),
    })
    pd.DataFrame(benchmark_records).to_csv(csv_output_path, index=False)


In [ ]:
# --- 11. Final Consolidated Benchmark Summary ---
df_results = pd.DataFrame(benchmark_records)
df_results = df_results.sort_values(by="Accuracy (%)", ascending=False).reset_index(drop=True)

print("=" * 100)
print("TRACK E: PRE-TRANSFORMER DEEP LEARNING BENCHMARK SUMMARY (BANKING77 FULL 10K, SEED 42)")
print("=" * 100)
display(df_results) if 'display' in dir() else print(df_results.to_string(index=False))

df_results.to_csv(csv_output_path, index=False)
print(f"\n[SAVED] Benchmark summary table exported to:\n   -> {csv_output_path}")


In [ ]:
# --- 12. Comparative Benchmark Visualizations ---
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, axes = plt.subplots(1, 2, figsize=(22, 6))

# Bar plot: Accuracy vs Macro F1
x_pos = np.arange(len(df_results))
width = 0.35
colors_acc = []
for name in df_results['Model Name']:
    if 'Ensemble' in name:
        colors_acc.append('#d62728')
    elif 'USE' in name:
        colors_acc.append('#9467bd')
    else:
        colors_acc.append('#1f77b4')

axes[0].bar(x_pos - width/2, df_results["Accuracy (%)"], width, label="Accuracy (%)", color=colors_acc)
axes[0].bar(x_pos + width/2, df_results["Macro F1 (%)"], width, label="Macro F1 (%)", color="#2ca02c", alpha=0.8)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(df_results["Model Name"], rotation=45, ha="right", fontsize=9)
axes[0].set_ylabel("Percentage (%)")
axes[0].set_title("Track E Architecture Performance Comparison (Including USE & Ensembles)", fontsize=12, fontweight="bold")
axes[0].legend()
axes[0].set_ylim([max(0, df_results["Accuracy (%)"].min() - 5), min(100, df_results["Accuracy (%)"].max() + 3)])

# Scatter plot: Accuracy vs Parameters
scatter_colors = []
for name in df_results['Model Name']:
    if 'Ensemble' in name:
        scatter_colors.append('#d62728')
    elif 'USE' in name:
        scatter_colors.append('#9467bd')
    else:
        scatter_colors.append('#1f77b4')

axes[1].scatter(df_results["Parameters"] / 1e6, df_results["Accuracy (%)"], s=120, c=scatter_colors, edgecolors="black")
for _, row in df_results.iterrows():
    axes[1].annotate(row["Model Name"], (row["Parameters"] / 1e6, row["Accuracy (%)"]),
                     textcoords="offset points", xytext=(5, 5), fontsize=8)
axes[1].set_xlabel("Model Parameters (Millions)")
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title("Accuracy vs. Model Size (Parameters)", fontsize=12, fontweight="bold")

plt.tight_layout()
plot_path = os.path.join(BASE_OUTPUT_DIR, "track_e_benchmark_performance.png")
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"[SAVED] Benchmark visualization saved to: {plot_path}")
